# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("master_cow_dataV2.csv")

In [ ]:
# frac=1 shuffles the entire dataset; random_state makes it reproducible
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
features = ['nose_neck_y_diff', 'neck_hip_y_diff', 'stride_length', 'tail_hip_dist', 'movement']
X = df_shuffled[features]
y = df_shuffled['behavior_label']

In [ ]:
# By default, train_test_split also shuffles, but shuffling the whole DF is safer
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Using 'balanced' weights to handle different class sizes
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Walking', 'Eating', 'Distress','Standing','lying']))

              precision    recall  f1-score   support

     Walking       0.95      0.72      0.82        58
      Eating       0.89      0.98      0.93       168
    Distress       0.97      0.92      0.94        90
    Standing       0.99      1.00      1.00       331
       lying       0.99      0.99      0.99       150

    accuracy                           0.96       797
   macro avg       0.96      0.92      0.94       797
weighted avg       0.97      0.96      0.96       797



In [ ]:
import joblib
joblib.dump(rf_model, 'final_cow_behavior_modelV2.pkl')

['final_cow_behavior_modelV2.pkl']

# Evaluating on new video

In [ ]:
import joblib

# 1. Load the Model and the Scaler
model = joblib.load('final_cow_behavior_modelV2.pkl')
scaler = joblib.load('cow_scaler.pkl')

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
new_video_df = pd.read_csv('/content/standing2_cow_behavior_features.csv')

In [ ]:
new_video_df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label
0,3,79.547241,194.307739,615.232910,62.141602,0.000000,0.642630,0.581965,NaN
1,4,21.270935,91.791687,233.715251,28.370117,23.645264,0.633107,0.579356,NaN
2,5,25.565715,90.326742,222.843750,33.069010,22.837809,0.620091,0.582501,NaN
3,6,42.849711,92.368958,265.171468,42.488688,70.449870,0.554529,0.594368,NaN
4,7,38.754985,103.210002,257.375488,32.468913,69.942301,0.590905,0.615096,NaN


In [ ]:
features = ['nose_neck_y_diff', 'neck_hip_y_diff', 'stride_length', 'tail_hip_dist', 'movement']
X_new = new_video_df[features]

In [ ]:
X_new_scaled = scaler.transform(X_new)

In [ ]:
new_video_df['behavior_label'] = model.predict(X_new_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
label_map = {0: 'Walking', 1: 'Eating', 2: 'Standing', 3: 'Distress', 4: 'Lying'}
new_video_df['behavior_name'] = new_video_df['behavior_label'].map(label_map)

In [ ]:
new_video_df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label,behavior_name
0,3,79.547241,194.307739,615.232910,62.141602,0.000000,0.642630,0.581965,2,Standing
1,4,21.270935,91.791687,233.715251,28.370117,23.645264,0.633107,0.579356,1,Eating
2,5,25.565715,90.326742,222.843750,33.069010,22.837809,0.620091,0.582501,2,Standing
3,6,42.849711,92.368958,265.171468,42.488688,70.449870,0.554529,0.594368,2,Standing
4,7,38.754985,103.210002,257.375488,32.468913,69.942301,0.590905,0.615096,2,Standing


In [ ]:
new_video_df.to_csv('new_video_results.csv', index=False)
print("Predictions complete! Check 'new_video_results.csv'")
print(new_video_df['behavior_name'].value_counts())

Predictions complete! Check 'new_video_results.csv'
behavior_name
Standing    246
Eating        4
Name: count, dtype: int64
